# LAM → OpenAvatarChat (OAC) avatar bake — Colab (free T4)

Bake **one** avatar from **one** portrait with [aigc3d/LAM](https://github.com/aigc3d/LAM) and export the
**OAC zip** (`skin.glb`, `offset.ply`, `animation.glb`, `vertex_order.json`) that
`ints-head-gs` loads. **Not** `h5_render_data.zip`.

**Before anything:** `Runtime ▸ Change runtime type ▸ T4 GPU`. Then run cells 1 → 7 top to bottom.

### Why a conda env (the Python-3.10 problem)
LAM's OAC export builds `skin.glb` with the **FBX SDK**, shipped **only as a cp310 wheel** → it needs
**Python 3.10**. Today's Colab is **Python 3.12**, and condacolab installs its own recent build (also 3.12),
so forcing a 3.10 *base* doesn't work. Instead, **Cell 1b creates a named conda env `lam` on Python 3.10**,
and every install/bake step runs inside it via `conda run -n lam`. No kernel restart; the env is deterministic.

> `files.upload()` / `files.download()` (Cells 5/7) run in the normal Colab kernel — they only move files on
> disk, so they don't need the env. Everything that imports LAM/torch/fbx goes through `conda run`.

### ⚠️ Honest status
Built from LAM's documented Linux install + source. The Python-3.10-base approach was tried on live Colab and
**failed** (condacolab gave 3.12); this `conda run` rewrite is the fix but is **itself not yet confirmed on a
full live run**. Biggest remaining risks: CUDA-compiled deps building inside the env (Cell 2), and the FBX
wheel importing in the env (Cell 4). **Zero-setup fallback:** LAM's
[ModelScope Space](https://www.modelscope.cn/studios/Damo_XR_Lab/LAM_Large_Avatar_Model) exports the OAC zip
server-side.


In [ ]:
# Cell 1 — GPU + CUDA + Python report.  SUCCESS: T4 shown, 'CUDA available: True'.
!nvidia-smi
import sys, torch  # kernel torch (Colab's) — only used here to check the GPU; the env gets its own torch.
print('kernel python', '%d.%d.%d' % sys.version_info[:3], '| torch', torch.__version__,
      '| torch CUDA', torch.version.cuda, '| available', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then Restart and run all.'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB')
print('Colab is Python 3.12; Cell 1b builds a Python-3.10 conda env for the FBX SDK. This is expected.')
# LAM-20K inference is light (~1.4s on A100) and fits T4 (~15GB). Host RAM (~12GB) is the tighter limit.


## Cell 1b — build the Python 3.10 conda env (`lam`)
Installs Miniforge to disk (no kernel restart) and creates env `lam` on Python 3.10. Defines `RUN` — the
`conda run` prefix used by every later install/bake cell. **Run this once before Cell 2.**

Why this works where condacolab didn't: we don't change the base/kernel Python at all. `conda create
python=3.10` resolves a real 3.10 interpreter, and the cp310 FBX wheel installs into it.


In [ ]:
# Cell 1b — Python 3.10 env.  SUCCESS: prints 'env python: 3.10.x' and 'RUN = ...'.  (~3-5 min)
import os
CONDA = '/usr/local/miniforge3/bin/conda'
ENV = 'lam'
if not os.path.exists(CONDA):
    !wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh
    !bash /tmp/miniforge.sh -b -p /usr/local/miniforge3
# create the 3.10 env if it isn't there yet
if not os.path.isdir(f'/usr/local/miniforge3/envs/{ENV}'):
    !{CONDA} create -y -n {ENV} python=3.10
# RUN = prefix for EVERY later package/install/bake command (streams output live)
RUN = f'{CONDA} run -n {ENV} --no-capture-output'
_v = !{CONDA} run -n {ENV} python -c "import sys;print('%d.%d.%d'%sys.version_info[:3])"
print('env python:', _v[-1] if _v else '??')
assert _v and _v[-1].startswith('3.10'), f'env is not 3.10: {_v}'
print('RUN =', RUN)


## Cell 2 — install the VERIFIED dependency set into the `lam` env
Locks in the set that produced a working bake. It does **not** use `requirements.txt`
(that pins `transformers==4.41.2` and `huggingface_hub==0.23.2`, which break this
pipeline). Instead it installs explicit, version-correct deps, all via `{RUN}` into the
3.10 env, with **numpy pinned to 1.23.0** through a pip constraints file.

Three non-obvious traps baked in (see RUNBOOK):
- **`transformers==4.40.0`** — newer (5.x) disables torch on 2.3.0.
- **`diffusers==0.31.0`** — 0.27 breaks on modern `huggingface_hub` (`cached_download`).
- **ashawkey's `diff-gaussian-rasterization`** — returns 4 values (LAM needs 4);
  graphdeco-inria's returns 2 → the `4-vs-2` ValueError. We uninstall any stray copy
  and force ashawkey's.

`pytorch3d` is the prebuilt cu121/pyt230 wheel (no 45-min build); only
`diff-gaussian-rasterization` (small) and the FaceBoxes Cython ext compile, both
timeout-guarded. Fail-loud: every step is `subprocess.run(check=True[,timeout])`.


In [ ]:
# Cell 2 — install the VERIFIED dep set into the lam env.  FAILS LOUDLY (check=True + timeouts).
# SUCCESS: 'IMPORTS_OK' then 'INSTALL DONE'.  (~10-15 min)
import os, subprocess, torch
def sh(cmd, timeout=None):
    print('>>>', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, timeout=timeout)  # non-zero/timeout -> cell ERRORS
cu = (torch.version.cuda or '')
CU = 'cu121' if cu.startswith('12') else 'cu118'
WHL = f'https://download.pytorch.org/whl/{CU}'
print('runtime CUDA', cu, '->', CU)
if not os.path.isdir('/content/LAM'):
    sh('git clone https://github.com/aigc3d/LAM.git /content/LAM')
%cd /content/LAM
# numpy pinned to 1.23.0 EVERYWHERE via a pip constraints file
open('/tmp/constraints.txt','w').write('numpy==1.23.0\n')
C = '-c /tmp/constraints.txt'
# 1) torch stack + xformers (wheels)
sh(f'{RUN} pip install {C} torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url {WHL}')
sh(f'{RUN} pip install {C} -U xformers==0.0.26.post1 --index-url {WHL}')
# 2) build deps must precede the source builds
sh(f'{RUN} pip install {C} "numpy==1.23.0" Cython ninja setuptools wheel')
# 3) pytorch3d = PREBUILT wheel (no ~45-min source build). cu121 only.
assert CU == 'cu121', 'prebuilt pytorch3d wheel is cu121-only; cu118 would need a source build.'
sh(f'{RUN} pip install {C} --no-cache-dir pytorch3d '
   f'-f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt230/download.html')
# 4) core LAM deps — the exact verified set (replaces requirements.txt).
#    transformers PINNED 4.40.0 (newer disables torch on 2.3.0);
#    diffusers PINNED 0.31.0 (0.27 breaks on modern huggingface_hub).
sh(f'{RUN} pip install {C} '
   'opencv-python-headless gradio omegaconf einops roma '
   'moviepy imageio imageio-ffmpeg tyro lpips face-alignment '
   'loguru scikit-image kornia matplotlib trimesh jaxtyping plyfile tensorboard '
   'transformers==4.40.0 diffusers==0.31.0 accelerate safetensors')
# 5) chumpy + pymcubes: build isolation OFF (build against the env numpy/Cython)
sh(f'{RUN} pip install {C} --no-build-isolation chumpy pymcubes')
# 6) RASTERIZER — MUST be ashawkey's (4 return values). inria's returns 2 -> ValueError.
sh(f'{RUN} pip uninstall -y diff-gaussian-rasterization diff_gaussian_rasterization || true')
os.environ['MAX_JOBS'] = '4'
sh(f'{RUN} pip install {C} --no-build-isolation -v '
   '"git+https://github.com/ashawkey/diff-gaussian-rasterization.git"', timeout=900)
# 7) nvdiffrast (NVlabs — the build verified live; fast, JIT-compiles at first render)
sh(f'{RUN} pip install {C} --no-build-isolation '
   '"git+https://github.com/NVlabs/nvdiffrast.git"', timeout=600)
# 8) FaceBoxesV2 Cython ext
sh(f'{RUN} sh -c "cd external/landmark_detection/FaceBoxesV2/utils && sh make.sh"', timeout=600)
# 9) IMPORTS_OK: bake-critical pkgs + entry points. MPLBACKEND=Agg + sys.path set
#    INSIDE the script (conda run doesn't reliably pass a shell env var or cwd), so
#    LAM's matplotlib import picks the headless backend and lam/tools resolve.
_imp_check = "\n".join([
    "import os",
    "os.environ['MPLBACKEND'] = 'Agg'",
    "import sys",
    "sys.path.insert(0, '/content/LAM')",
    "import matplotlib",
    "matplotlib.use('Agg')",
    "import torch, pytorch3d, diff_gaussian_rasterization, nvdiffrast.torch",
    "from lam.models import ModelLAM",
    "from tools.flame_tracking_single_image import FlameTrackingSingleImage",
    "print('IMPORTS_OK')",
]) + "\n"
open('/tmp/imp_check.py', 'w').write(_imp_check)
sh(f'{RUN} python /tmp/imp_check.py', timeout=600)
print('INSTALL DONE')


## Cell 3 — download weights + assets (HuggingFace)
`3DAIGC/LAM-20K` + `3DAIGC/LAM-assets` are **public** — token optional (helps rate limits). Paste yours in the
placeholder; **do not commit it**. (`hf` runs in the env so it matches the installed hub version.)


In [ ]:
# Cell 3 — weights + assets + OAC template.  SUCCESS: 'WEIGHTS + ASSETS OK'.
# >>> OPTIONAL: paste your HF token (else anonymous). NEVER COMMIT THIS. <<<
HF_TOKEN = ''  # e.g. 'hf_xxx'
import os
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN  # snapshot_download reads this env var
# Download via the Python API (version-agnostic) — avoids depending on the `hf` /
# `huggingface-cli` CLI name, which differs across huggingface_hub versions.
open('/tmp/dl.py','w').write(
    'from huggingface_hub import snapshot_download\n'
    'snapshot_download("3DAIGC/LAM-assets", local_dir="./tmp")\n'
    'snapshot_download("3DAIGC/LAM-20K", local_dir="./model_zoo/lam_models/releases/lam/lam-20k/step_045500/")\n'
    'print("HF_DOWNLOAD_OK")\n')
!{RUN} python /tmp/dl.py
!tar -xf ./tmp/LAM_assets.tar && rm ./tmp/LAM_assets.tar
!tar -xf ./tmp/thirdparty_models.tar && rm -r ./tmp/
# OAC template — template_file.fbx lives in sample_oac.tar; the GLB export needs it.
!wget -q https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/sample_oac.tar -O ./sample_oac.tar
!tar -xf ./sample_oac.tar -C assets/ && rm ./sample_oac.tar
assert os.path.exists('model_zoo/lam_models/releases/lam/lam-20k/step_045500/model.safetensors'), 'LAM-20K weights missing'
assert os.path.exists('assets/sample_oac/template_file.fbx'), 'assets/sample_oac/template_file.fbx missing (OAC GLB export needs it)'
print('WEIGHTS + ASSETS OK')


## Cell 4 — Blender (headless) + FBX SDK (into the env)
`skin.glb` = ASCII FBX → *(FBX SDK)* binary FBX → *(Blender)* GLB; that Blender step also writes
`vertex_order.json`. Blender is a standalone binary (kernel-agnostic); the **cp310 FBX wheel installs into the
`lam` env** — which is the whole reason for Cell 1b.


In [ ]:
# Cell 4 — Blender + FBX SDK.  SUCCESS: 'BLENDER OK' AND 'import fbx OK ...'.
import os, subprocess, zipfile
BV = 'blender-4.0.2-linux-x64'  # guide-pinned; OAC needs Blender > 4.0
if not os.path.isdir(f'/content/{BV}'):
    !wget -q https://download.blender.org/release/Blender4.0/{BV}.tar.xz -O /content/blender.tar.xz
    !tar -xf /content/blender.tar.xz -C /content/
BLENDER = f'/content/{BV}/blender'
os.environ['BLENDER'] = BLENDER
r = os.system(f'{BLENDER} --background --version')
if r != 0:  # headless Blender still needs a few X libs present
    !apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libsm6 >/dev/null
    r = os.system(f'{BLENDER} --background --version')
assert r == 0, 'Blender headless failed — check the apt libs above.'
print('BLENDER OK:', BLENDER)

# OAC export helpers (guide Step1): pathlib backport + patool archive tool
!{RUN} pip install -q pathlib patool

# FBX SDK must import in the 3.10 env. If it already does, SKIP the wheel entirely.
def env_has_fbx():
    return subprocess.run(f'{CONDA} run -n {ENV} python -c "import fbx"', shell=True).returncode == 0

if env_has_fbx():
    print('import fbx OK (already in 3.10 env) — skipping wheel install')
else:
    # keep the REAL wheel filename — pip rejects a renamed 'fbx.whl' (needs version/tags)
    FBX_URL = 'https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/fbx-2020.3.4-cp310-cp310-manylinux1_x86_64.whl'
    FBX_WHL = '/content/' + FBX_URL.split('/')[-1]
    !wget -q {FBX_URL} -O {FBX_WHL}
    assert os.path.exists(FBX_WHL) and os.path.getsize(FBX_WHL) > 100_000 and zipfile.is_zipfile(FBX_WHL), \
        f'fbx wheel download invalid: {FBX_WHL} ({os.path.getsize(FBX_WHL) if os.path.exists(FBX_WHL) else 0} bytes) — bad URL?'
    assert os.system(f'{RUN} pip install -q {FBX_WHL}') == 0, 'fbx wheel install failed'
    assert env_has_fbx(), 'fbx still not importable after wheel install'
    print('import fbx OK (installed wheel in 3.10 env)')


---
# FLAME ARKit-52 bake — the teeth + blendshape path  ⬅️ alternate to OAC below

Everything above (Cells 1–4) is **shared env setup** — run those first either way.
This section is the **second, independent bake path**. The OAC path further down
produces the no-teeth `skin.glb/offset.ply/animation.glb/vertex_order.json` zip.
**This path** produces the **FLAME web zip WITH teeth and a full ARKit-52
blendshape set** — named morphs for **blink / emotion / jaw** (`eyeBlinkLeft`,
`eyeBlinkRight`, `jawOpen`, … the 52 ARKit channels), driven at runtime.

**What it's for / where it loads:**
- Output zip: `output/flame_render/<image>_arkit.zip` (e.g. **`fisherman_arkit.zip`**).
- Loads in **`flame-spike.html`** and **`compare.html` (v2)** in `ints-head-gs`.
- ⚠️ **Renderer patch required:** the web renderer only walks the FLAME code path
  when the **`__FORCE_FLAME`** patch is applied — without it the loader falls back
  to the OAC/Gaussian path and the ARKit morphs won't drive. Apply `__FORCE_FLAME`
  before testing this zip, or it'll look like the bake "did nothing".

Run order for this path: **A → B → C → D → E** below (after Cells 1–4).

### FLAME-A — pull the latest FLAME runner
Fetches `colab_bake_flame.py` from `main` into the cloned LAM repo, so this notebook
always bakes with the current runner (don't rely on a stale copy).

In [ ]:
# FLAME-A — pull the latest FLAME runner into /content/LAM.  SUCCESS: 'saved ... colab_bake_flame.py'.
!cd /content/LAM && wget -O colab_bake_flame.py https://raw.githubusercontent.com/IntsI/ints-head-gs/main/tools/colab_bake_flame.py

### FLAME-B — preflight (dependency guard)
`preflight_flame.py` checks the FLAME/ARKit deps + assets are present **before** you
spend a bake (exit 0 = good to go; non-zero prints exactly what's missing). Pulled
fresh alongside the runner. Same `--blender_path` / `--motion` you'll bake with.

In [ ]:
# FLAME-B — preflight the FLAME deps.  SUCCESS: exit 0, no FAIL lines.
!cd /content/LAM && wget -O preflight_flame.py https://raw.githubusercontent.com/IntsI/ints-head-gs/main/tools/preflight_flame.py
!MPLBACKEND=Agg {RUN} bash -c "cd /content/LAM && python preflight_flame.py --blender_path {BLENDER} --motion Look_In_My_Eyes"

### FLAME-B.5 — shape-dim probe (which dims = face length / width / chin)
Reads the **actual** `flame2023.pkl` shape basis and ranks dims 0–10 by their real
effect on face **length / width / chin / jaw**, so `--shape-edit` is data-driven, not
guesswork. Run after Cell 3 (needs `model_zoo/.../flame2023.pkl`).

**Measured on this basis (FLAME2023):**
- **dim1 = elongation** — `+length, −width, −jaw`. The clean *longer + narrower* lever → push **positive**.
- **dim0 = overall size / roundness** — biggest `+width`/`+jaw`/`+chin` driver → push **negative** to de-round.
- dims 2,4,6–10 are weak/mixed — skip.

**For a round/soft face → longer & more angular:** `dim1 ↑` primary, `dim0 ↓` removes roundness.
Ready-to-paste (pass to FLAME-D's `--shape-edit`, with a matching `--tag`):
- subtle longer: `1:+1.5`
- **longer + narrower (recommended):** `1:+1.8,0:-1.0`
- max angular (long, narrow, tapered jaw): `2.0` → `1:+2.0,0:-1.2,3:+0.8`

**Safe magnitude:** betas are unit-σ PCA coeffs, effect is linear; start **±1.0–1.5**, ceiling **~±2.5**;
past ±3 it extrapolates out of distribution → distortion. (Probe measures the base
un-subdivided template, so direction/ranking are exact; rendered mm shift slightly with teeth/subdiv.)

In [ ]:
%%writefile /content/LAM/probe_flame_dims.py
# FLAME shape-dim probe — reads the ACTUAL flame2023.pkl basis and ranks shape
# dims 0-10 by their real effect on face LENGTH / WIDTH / CHIN / JAW. Use the
# printed dims to build --shape-edit strings (e.g. dim1=elongate, dim0=size).
# Robust loader: a stub unpickler pulls the ndarrays straight out of the chumpy
# objects, so it works whether or not chumpy imports cleanly.
import pickle, glob, os, numpy as np
np.set_printoptions(suppress=True)

# locate the basis the FLAME runner uses (downloaded into model_zoo by Cell 3 /
# preflight). Prefer flame_assets/flame to match build_arkit_flame_model().
def find(name):
    pats = [f"./model_zoo/**/flame_assets/flame/{name}",
            f"/content/LAM/model_zoo/**/flame_assets/flame/{name}",
            f"./model_zoo/**/{name}", f"/content/**/{name}"]
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(f"{name} not found under model_zoo — run Cell 3 (assets) first.")

class Carrier:
    def __setstate__(self, st): self.__dict__["_st"] = st
    def __reduce__(self): return (Carrier, ())
class SU(pickle.Unpickler):
    def find_class(self, m, n):
        if m.startswith("chumpy") or m.startswith("scipy"): return Carrier
        return super().find_class(m, n)
def load(f): return SU(open(f, "rb"), encoding="latin1", fix_imports=True).load()
def arr(v):
    if isinstance(v, Carrier):
        st = v.__dict__.get("_st")
        if isinstance(st, dict): return st.get("x", st.get("r"))
    return v

B = find("flame2023.pkl"); M = find("FLAME_masks.pkl")
print("basis:", B)
d = load(B)
vt = np.asarray(arr(d["v_template"]), np.float64)        # [V,3]
sd = np.asarray(arr(d["shapedirs"]), np.float64)[:, :, :300]   # shape PCA (betas index here)
V = vt.shape[0]
masks = load(M)

# axes, data-driven: width = mirror-symmetric axis; length = larger remaining extent
from scipy.spatial import cKDTree
def sym(ax):
    p = vt.copy(); p[:, ax] *= -1
    return cKDTree(vt).query(p)[0].mean()
res = [sym(a) for a in range(3)]
W = int(np.argmin(res)); ext = vt.max(0) - vt.min(0)
rest = [a for a in range(3) if a != W]
Vax = rest[int(np.argmax(ext[rest]))]; Dax = [a for a in rest if a != Vax][0]
def idx(name):
    x = masks.get(name) if isinstance(masks, dict) else None
    return np.asarray(arr(x)).astype(int).ravel() if x is not None else None
face = idx("face"); face = np.arange(V) if face is None else face
fh = idx("forehead"); up = 1.0
if fh is not None: up = np.sign(vt[fh][:, Vax].mean() - vt[face][:, Vax].mean()) or 1.0
print(f"sym_residual/axis={np.round(res,4)}  extents={np.round(ext,3)}")
print(f"=> WIDTH=axis{W}  LENGTH=axis{Vax}  DEPTH=axis{Dax}  up_sign={up:+.0f}")
chin = face[np.argsort(vt[face][:, Vax] * up)[:max(20, len(face)//50)]]
jaw = face[np.argsort(vt[face][:, Vax] * up)[:max(60, len(face)//6)]]
def ea(p, a): return p[:, a].max() - p[:, a].min()
bl, bw = ea(vt, Vax), ea(vt, W)
print(f"base length(axis{Vax})={bl:.3f}  width(axis{W})={bw:.3f}\n")
print(f"{'dim':>3} {'Dlength':>9} {'Dwidth':>9} {'chinDrop':>9} {'jawWidth':>9}  amp")
rows = []
for k in range(11):
    nv = vt + sd[:, :, k]                                  # +1 sigma of beta dim k
    dl = ea(nv, Vax) - bl; dw = ea(nv, W) - bw
    dc = -(nv[chin][:, Vax] - vt[chin][:, Vax]).mean() * up   # +=chin lower (longer)
    jw = ea(nv[jaw], W) - ea(vt[jaw], W); amp = np.abs(sd[:, :, k]).max()
    rows.append((k, dl, dw, dc, jw, amp))
    print(f"{k:>3} {dl:>+9.4f} {dw:>+9.4f} {dc:>+9.4f} {jw:>+9.4f}  {amp:.3f}")
def rank(c, l):
    s = sorted(rows, key=lambda r: -abs(r[c]))[:4]
    print(f"{l}: " + ", ".join(f"dim{r[0]}({r[c]:+.4f})" for r in s))
print()
rank(1, "LENGTH +=longer "); rank(2, "WIDTH  +=wider  ")
rank(3, "CHIN   +=longer "); rank(4, "JAWWID +=wider  ")
print("\nbuild --shape-edit from the LENGTH/WIDTH dims: push the elongation dim +,")
print("the size/width dim - for a longer, less-round face (see FLAME-B.5 notes).")


In [ ]:
# FLAME-B.5 (run) — print the dim ranking.  SUCCESS: a dim table + LENGTH/WIDTH/CHIN ranks.
!{RUN} python /content/LAM/probe_flame_dims.py

### FLAME-C — upload your portrait
Same as the OAC upload cell — included here so this path is self-contained. Saves to
`assets/sample_input/fisherman.jpg`. If you already ran the OAC upload cell with the
same image, you can skip this. Front-facing, well-lit works best.

In [ ]:
# FLAME-C — upload your portrait (Colab kernel — no env needed).
# SUCCESS: 'Saved: assets/sample_input/fisherman.jpg'.
from google.colab import files
import os, shutil
up = files.upload()  # choose your image
src = list(up.keys())[0]
os.makedirs('/content/LAM/assets/sample_input', exist_ok=True)
IMG = '/content/LAM/assets/sample_input/fisherman.jpg'
shutil.move(src, IMG)
print('Saved:', IMG)

### FLAME-D — the ARKit bake (the working command)
`--arkit` builds the 52-channel ARKit→FLAME head (teeth on by default; add `--no-teeth`
to drop them). `--tag arkit` names the output `fisherman_arkit.zip`. `--motion
Look_In_My_Eyes` supplies the driving FLAME params. `MPLBACKEND=Agg` keeps matplotlib
headless; the `bash -c "cd …"` wrapper guarantees the runner sees the repo as cwd
under `conda run`. Swap `fisherman.jpg` for your uploaded image name if different.

In [ ]:
# FLAME-D — run the ARKit bake.  SUCCESS: '✅ FLAME zip ready: .../output/flame_render/fisherman_arkit.zip'.
!MPLBACKEND=Agg {RUN} bash -c "cd /content/LAM && python colab_bake_flame.py --image assets/sample_input/fisherman.jpg --blender_path {BLENDER} --motion Look_In_My_Eyes --arkit --tag arkit"

### FLAME-E — download `fisherman_arkit.zip`
Pulls the FLAME zip to your Mac. Drop it into `ints-head-gs` and open
`flame-spike.html` / `compare.html` (v2) — **with the `__FORCE_FLAME` patch applied**
— to see blink / emotion / jaw drive via the named ARKit morphs.

In [ ]:
# FLAME-E — download the FLAME ARKit zip (Colab kernel).  SUCCESS: browser download of fisherman_arkit.zip.
from google.colab import files
import glob, os
zips = sorted(glob.glob('/content/LAM/output/flame_render/*.zip'), key=os.path.getmtime)
assert zips, 'no FLAME zip found — the FLAME-D bake cell did not complete.'
print('downloading', zips[-1])  # e.g. fisherman_arkit.zip
files.download(zips[-1])

---
# OAC bake path (no teeth)  ⬇️
The original OAC export below is unchanged. It produces the no-teeth OAC zip
(`skin.glb`, `offset.ply`, `animation.glb`, `vertex_order.json`). Use **either** this
OAC path **or** the FLAME-ARKit path above — both run on the same Cells 1–4 env.

In [ ]:
# Cell 5 — upload your fisherman portrait (runs in the Colab kernel — no env needed).
# SUCCESS: prints 'Saved: assets/sample_input/fisherman.jpg'. Front-facing, well-lit works best.
from google.colab import files
import os, shutil
up = files.upload()  # choose your image
src = list(up.keys())[0]
os.makedirs('assets/sample_input', exist_ok=True)
IMG = 'assets/sample_input/fisherman.jpg'
shutil.move(src, IMG)
print('Saved:', IMG)


## Cell 6 — bake + OAC export (headless, in the env)
Writes the gradio-free runner (adapted from `app_lam.py core_fn`) and runs it **via `conda run`** so it
executes on Python 3.10 with the FBX SDK + LAM deps. Output: `output/open_avatar_chat/<stem>.zip`
(inner folder == zip name). If it raises on a LAM internal, use the Gradio fallback below.

**Teeth (experimental):** add `--add-teeth` (with e.g. `--tag teeth`) to flip
`cfg.model.add_teeth=True`. It runs the full bake+export and surfaces the real
failure if the no-teeth `template_file.fbx` desyncs (prints the vertex counts).


In [ ]:
%%writefile colab_bake_oac.py
#!/usr/bin/env python3
"""
Headless LAM → OpenAvatarChat (OAC) bake — one image in, one OAC zip out.

This is the gradio-free path used by the Colab notebook (LAM_bake_oac_colab.ipynb).
It is adapted *faithfully* from `app_lam.py`'s `core_fn` OAC-export branch
(the part gated behind the "Export ZIP file for Chatting Avatar" checkbox), with
the video-rendering steps stripped out — we only need the four OAC files.

Run from inside the cloned LAM repo:
    python colab_bake_oac.py \
        --image assets/sample_input/fisherman.jpg \
        --blender_path /content/blender-4.0.2-linux-x64/blender \
        --motion auto

Output: ./output/open_avatar_chat/<image-stem>.zip containing
    <image-stem>/skin.glb, offset.ply, animation.glb, vertex_order.json
i.e. exactly LAM's OAC format (see ints-head-gs/docs/AVATAR_FORMAT.md). NOT
h5_render_data.zip.

⚠️ This orchestration mirrors LAM internals at a point in time. If LAM changes a
signature (infer_single_view / prepare_motion_seqs / save_shaped_mesh), this will
raise — fall back to the notebook's Gradio cell, which runs LAM's own code.
"""
import argparse
import os
import sys

# Robustness (conda run doesn't reliably pass a shell-prefix env var or cwd to
# python): force the headless matplotlib backend BEFORE lam imports matplotlib,
# and put the repo on sys.path so `lam`/`tools` resolve regardless of cwd.
os.environ["MPLBACKEND"] = "Agg"
sys.path.insert(0, "/content/LAM")

import shutil
import zipfile
from glob import glob
from pathlib import Path


def find_motion(motion_arg: str) -> str:
    """Pick a driving motion-sequence dir (provides flame shape + render params)."""
    if motion_arg and motion_arg != "auto":
        d = f"./assets/sample_motion/export/{motion_arg}"
        assert os.path.isdir(d), f"motion not found: {d}"
        return d
    cands = sorted(glob("./assets/sample_motion/export/*/"))
    assert cands, "no sample motions under assets/sample_motion/export/ — did the assets download succeed?"
    # prefer the one LAM's own inference.sh uses, if present
    for c in cands:
        if "Look_In_My_Eyes" in c:
            return c.rstrip("/")
    return cands[0].rstrip("/")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--image", required=True, help="input portrait, e.g. assets/sample_input/fisherman.jpg")
    ap.add_argument("--blender_path", required=True, help="path to Blender >4.0 executable")
    ap.add_argument("--motion", default="auto", help="motion seq name under assets/sample_motion/export, or 'auto'")
    ap.add_argument("--shape-edit", dest="shape_edit", default="",
                    help="warp FLAME shape dims as deltas on the fitted shape, e.g. "
                         "'0:+1.5,3:-1.0'. Abstract FLAME PCA dims (not nose/jaw). "
                         "Same edit must be reused to reproduce a variant; bake several "
                         "to compare in the spike's variant switcher.")
    ap.add_argument("--tag", default="",
                    help="suffix for the output name so variants don't collide, e.g. "
                         "--tag s0plus -> <image>_s0plus.zip (distinct zip + inner folder "
                         "+ switcher chip). Use a different tag per variant bake.")
    ap.add_argument("--add-teeth", dest="add_teeth", action="store_true",
                    help="EXPERIMENTAL: enable FLAME teeth (cfg.model.add_teeth=True). Predicted to "
                         "desync the OAC export (no-teeth template_file.fbx is hardcoded to a fixed "
                         "vertex count). This runs the full bake+export anyway so the real failure is "
                         "visible — counts are printed and the error is surfaced, not hidden.")
    args = ap.parse_args()

    assert os.path.exists(args.image), f"image not found: {args.image}"
    assert os.path.exists(args.blender_path), f"blender not found: {args.blender_path}"

    # --- env + config, copied from app_lam.launch_gradio_app -----------------
    os.environ.update({
        "APP_ENABLED": "1",
        "APP_MODEL_NAME": "./model_zoo/lam_models/releases/lam/lam-20k/step_045500/",
        "APP_INFER": "./configs/inference/lam-20k-8gpu.yaml",
        "APP_TYPE": "infer.lam",
        "NUMBA_THREADING_LAYER": "omp",
    })

    import matplotlib
    matplotlib.use("Agg")  # belt-and-suspenders with the MPLBACKEND env set at module top
    import torch
    import app_lam  # importing does NOT launch gradio (that's under __main__)
    from tools.generateARKITGLBWithBlender import generate_glb
    from lam.runners.infer.head_utils import prepare_motion_seqs, preprocess_image

    # parse_configs reads --blender_path off sys.argv; hand it a clean argv.
    sys.argv = ["colab_bake_oac.py", "--blender_path", args.blender_path]
    cfg, _ = app_lam.parse_configs()

    # EXPERIMENTAL teeth: override the config flag (configs/inference/lam-20k-8gpu.yaml:43
    # add_teeth: false) -> flows to gs_renderer -> FlameHeadSubdivided. Changes the FLAME
    # vertex/gaussian count, which the no-teeth OAC template_file.fbx isn't built for.
    if args.add_teeth:
        cfg.model.add_teeth = True
        print("⚠️  --add-teeth: cfg.model.add_teeth=True (experimental; OAC export may desync).")

    print("building model + flame tracking…")
    lam = app_lam._build_model(cfg)
    lam.to("cuda").eval()

    from tools.flame_tracking_single_image import FlameTrackingSingleImage
    flametracking = FlameTrackingSingleImage(
        output_dir="output/tracking",
        alignment_model_path="./model_zoo/flame_tracking_models/68_keypoints_model.pkl",
        vgghead_model_path="./model_zoo/flame_tracking_models/vgghead/vgg_heads_l.trcd",
        human_matting_path="./model_zoo/flame_tracking_models/matting/stylematte_synth.pt",
        facebox_model_path="./model_zoo/flame_tracking_models/FaceBoxesV2.pth",
        detect_iris_landmarks=False,
    )

    clip_dir = find_motion(args.motion)
    # LAM's core_fn passes the clip's `flame_param` subdir; prepare_motion_seqs then
    # reads transforms.json from its PARENT (the clip dir). Passing the clip dir
    # itself makes it look for transforms.json one level too high.
    motion_seqs_dir = os.path.join(clip_dir, "flame_param")
    assert os.path.isdir(motion_seqs_dir), f"missing flame_param dir: {motion_seqs_dir}"
    base_iid = os.path.basename(args.image).split(".")[0]
    if args.tag.strip():
        # keep names filesystem/URL-safe so variants get distinct zip + folder + chip
        safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in args.tag.strip())
        base_iid = f"{base_iid}_{safe}"
    print(f"image={args.image}  iid={base_iid}  clip={clip_dir}  motion_seqs_dir={motion_seqs_dir}")

    # --- flame tracking on the input image (core_fn steps) -------------------
    tmp_dir = "output/_bake_tmp"
    os.makedirs(tmp_dir, exist_ok=True)
    image_raw = os.path.join(tmp_dir, "raw.png")
    from PIL import Image
    with Image.open(args.image).convert("RGB") as im:
        im.save(image_raw)

    assert flametracking.preprocess(image_raw) == 0, "flametracking preprocess failed"
    assert flametracking.optimize() == 0, "flametracking optimize failed"
    rc, output_dir = flametracking.export()
    assert rc == 0, "flametracking export failed"

    image_path = os.path.join(output_dir, "images/00000_00.png")
    mask_path = os.path.join(output_dir, "fg_masks/00000_00.png")

    aspect_standard = 1.0 / 1.0
    image, _, _, shape_param = preprocess_image(
        image_path, mask_path=mask_path, intr=None, pad_ratio=0, bg_color=1.0,
        max_tgt_size=None, aspect_standard=aspect_standard, enlarge_ratio=[1.0, 1.0],
        render_tgt_size=cfg.source_size, multiply=14, need_mask=True, get_shape_param=True,
    )

    # --- optional shape warp (applied to the fitted betas, BEFORE all consumers) ---
    # Must run before prepare_motion_seqs/infer/save_shaped_mesh so the gaussians AND
    # the skin.glb mesh get the same modified shape (else they desync).
    if args.shape_edit.strip():
        edits = []
        for tok in args.shape_edit.split(","):
            if not tok.strip():
                continue
            dim_s, delta_s = tok.split(":")
            dim, delta = int(dim_s), float(delta_s)
            assert 0 <= dim < shape_param.shape[-1], f"shape dim {dim} out of range (0..{shape_param.shape[-1]-1})"
            shape_param[dim] += delta
            edits.append(f"dim{dim}{delta:+g}")
        print(f"shape-edit applied to fitted betas: {', '.join(edits)}")

    src = image_path.split("/")[-3]
    driven = motion_seqs_dir.split("/")[-2]
    motion_seq = prepare_motion_seqs(
        motion_seqs_dir, None, save_root=tmp_dir, fps=30, bg_color=1.0,
        aspect_standard=aspect_standard, enlarge_ratio=[1.0, 1, 0],
        render_image_res=cfg.render_size, multiply=16, need_mask=False,
        vis_motion=False, shape_param=shape_param, test_sample=False,
        cross_id=False, src_driven=[src, driven],
    )

    # --- inference → canonical gaussians -------------------------------------
    device, dtype = "cuda", torch.float32
    motion_seq["flame_params"]["betas"] = shape_param.unsqueeze(0)
    print("running LAM inference…")
    with torch.no_grad():
        res = lam.infer_single_view(
            image.unsqueeze(0).to(device, dtype), None, None,
            render_c2ws=motion_seq["render_c2ws"].to(device),
            render_intrs=motion_seq["render_intrs"].to(device),
            render_bg_colors=motion_seq["render_bg_colors"].to(device),
            flame_params={k: v.to(device) for k, v in motion_seq["flame_params"].items()},
        )

    # --- OAC export (app_lam.py lines ~304-342, minus the video) -------------
    oac_dir = os.path.join("./output/open_avatar_chat", base_iid)
    os.makedirs(oac_dir, exist_ok=True)
    print("writing offset.ply…")
    saved_head_path = lam.renderer.flame_model.save_shaped_mesh(
        shape_param.unsqueeze(0).cuda(), fd=oac_dir,
    )
    res["cano_gs_lst"][0].save_ply(os.path.join(oac_dir, "offset.ply"), rgb2sh=False, offset2xyz=True)

    template_fbx = "./assets/sample_oac/template_file.fbx"

    # Empirical teeth diagnostic: report the mesh vs template vertex counts (the
    # predicted desync) BEFORE attempting the export — informational, not an abort.
    if args.add_teeth:
        try:
            obj_verts = sum(1 for ln in open(saved_head_path) if ln.startswith("v "))
            import re as _re
            m = _re.search(r"Vertices:\s*\*(\d+)", open(template_fbx, encoding="utf-8", errors="ignore").read())
            tmpl_floats = int(m.group(1)) if m else None
            print("=" * 60)
            print(f"[add-teeth] shaped mesh (nature.obj): {obj_verts} verts = {obj_verts*3} floats")
            print(f"[add-teeth] template_file.fbx declares: Vertices: *{tmpl_floats}"
                  + (" floats" if tmpl_floats else " (not found)"))
            if tmpl_floats is not None:
                match = (obj_verts * 3 == tmpl_floats)
                print(f"[add-teeth] vertex counts {'MATCH' if match else 'MISMATCH'} "
                      f"({obj_verts*3} vs {tmpl_floats}) — "
                      + ("export should be fine" if match else "export likely to fail/desync below"))
            print("=" * 60)
        except Exception as e:  # diagnostic only — never block the real attempt
            print(f"[add-teeth] count diagnostic skipped: {e}")

    print("generating skin.glb via Blender + FBX SDK (also writes vertex_order.json)…")
    try:
        generate_glb(
            input_mesh=Path(saved_head_path),
            template_fbx=Path(template_fbx),
            output_glb=Path(os.path.join(oac_dir, "skin.glb")),
            blender_exec=Path(cfg.blender_path),
        )
    except Exception as e:
        if args.add_teeth:
            print("\n" + "!" * 60)
            print("[add-teeth] OAC export FAILED at generate_glb (skin.glb step).")
            print("[add-teeth] This is the predicted teeth desync: the shaped mesh now has")
            print("[add-teeth] teeth verts, but template_file.fbx is hardcoded to the no-teeth")
            print("[add-teeth] count (see Vertices: *N above). A teeth-aware template_file.fbx")
            print("[add-teeth] would be needed. Full error follows:")
            print("!" * 60)
        raise  # surface the real failure — do not hide it
    shutil.copy("./assets/sample_oac/animation.glb", os.path.join(oac_dir, "animation.glb"))
    if os.path.exists(saved_head_path):
        os.remove(saved_head_path)

    # --- validate the 4 OAC files, then zip with inner-folder == zip-name ----
    required = ["skin.glb", "offset.ply", "animation.glb", "vertex_order.json"]
    missing = [f for f in required if not os.path.exists(os.path.join(oac_dir, f))]
    assert not missing, f"OAC export incomplete, missing: {missing} (vertex_order.json comes from generate_glb step 4)"

    out_zip = os.path.join("./output/open_avatar_chat", base_iid + ".zip")
    if os.path.exists(out_zip):
        os.remove(out_zip)
    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
        # The renderer finds the avatar folder by scanning for a DIRECTORY ENTRY
        # (dir==true). Python's zipfile writes only file entries by default, so we
        # must add the directory entry explicitly or the loader throws
        # 'file fold is not found'. (See tools/repack_oac.py / docs/AVATAR_FORMAT.md.)
        di = zipfile.ZipInfo(base_iid + "/")
        di.external_attr = (0o40755 << 16) | 0x10  # unix dir bit + MS-DOS dir flag
        z.writestr(di, b"")
        for f in required:
            z.write(os.path.join(oac_dir, f), arcname=os.path.join(base_iid, f))

    print("\n✅ OAC zip ready:", os.path.abspath(out_zip))
    print("   inner folder:", base_iid, "(with directory entry; matches docs/AVATAR_FORMAT.md)")
    print("   contents:", ", ".join(required))


if __name__ == "__main__":
    main()


In [ ]:
# Cell 6 (run).  SUCCESS: '\u2705 OAC zip ready: .../output/open_avatar_chat/fisherman.zip'.
# MPLBACKEND=Agg: matplotlib's default inline backend crashes headless.
# --motion takes the CLIP NAME only (find_motion prepends assets/sample_motion/export/).
!MPLBACKEND=Agg {RUN} python colab_bake_oac.py --image assets/sample_input/fisherman.jpg --blender_path {BLENDER} --motion Look_In_My_Eyes


### Cell 6 — Gradio fallback (only if the headless runner errors)
Runs LAM's exact UI code in the env. Open the printed public URL → upload your image → pick a driving video
example → **tick 'Export ZIP file for Chatting Avatar'** → Generate. Zip lands in `output/open_avatar_chat/`.


In [ ]:
# Optional fallback — LAM's gradio app with a public share link, in the env.
# !sed -i 's/demo.launch()/demo.launch(share=True)/' app_lam.py
# !{RUN} python app_lam.py --blender_path {BLENDER}


In [ ]:
# Cell 7 — download the OAC zip to your Mac (Colab kernel).  SUCCESS: browser download of the zip.
from google.colab import files
import glob, os
zips = sorted(glob.glob('output/open_avatar_chat/*.zip'), key=os.path.getmtime)
assert zips, 'no OAC zip found — Cell 6 did not complete.'
print('downloading', zips[-1])
files.download(zips[-1])


---
### Verify before baking a batch
Drop the downloaded zip into `ints-head-gs`: `http://localhost:5173/?avatar=<url>` or the drag-drop zone
(validates the 4 files + folder-name rule per `docs/AVATAR_FORMAT.md`). Confirm **one** avatar renders first.
